**Calorie data cleaning - July 9, 2026**

In [1]:
%pip install pandas matplotlib numpy scikit-learn psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
#importing the database
from src.database_service import DatabaseService


In [4]:
db = DatabaseService() 

In [5]:
dir(db)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'connect',
 'connection',
 'dbname',
 'disconnect',
 'extract_from_database',
 'get_audio_records',
 'host',
 'output_dir',
 'password',
 'port',
 'remove_test_users',
 'user']

In [6]:
db.connect() #connecting to the DB

True

app_user_id 1, 2, 3, 10, 43 are test cases and are automatically removed from the DB.

In [7]:
tables = db.extract_from_database("information_schema.tables")
tables

/Users/pegah/Projects/lemurs-mqp/src/database_service.py:194: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.connection)
/Users/pegah/Projects/lemurs-mqp/src/database_service.py:218: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_users_df = pd.read_sql(query, self.connection)
2026-07-08 18:29:39,804 - INFO - Identified 5 test users to remove


,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,lemurs,public,app_user,BASE TABLE,None,None,None,None,None,YES,NO,None
1,lemurs,public,umass_id,BASE TABLE,None,None,None,None,None,YES,NO,None
2,lemurs,public,user_info,BASE TABLE,None,None,None,None,None,YES,NO,None
3,lemurs,public,auth_microsoft,BASE TABLE,None,None,None,None,None,YES,NO,None
4,lemurs,public,authorized_email,BASE TABLE,None,None,None,None,None,YES,NO,None
...,...,...,...,...,...,...,...,...,...,...,...,...
243,lemurs,public,danger_alert,BASE TABLE,None,None,None,None,None,YES,NO,None
244,lemurs,public,all_danger_alert,VIEW,None,None,None,None,None,NO,NO,None
245,lemurs,public,missing_daily_surveys,VIEW,None,None,None,None,None,NO,NO,None
246,lemurs,public,danger_alert_trigger,BASE TABLE,None,None,None,None,None,YES,NO,None


In [8]:
#looking at the table names
tables["table_name"].tolist()

['app_user',
 'umass_id',
 'user_info',
 'auth_microsoft',
 'authorized_email',
 'authorized_email_elevated',
 'app_role',
 'data',
 'pg_statistic',
 'pg_type',
 'danger_alert_email',
 'question',
 'survey',
 'survey_question',
 'survey_question_view',
 'pg_foreign_table',
 'survey_response',
 'pg_authid',
 'pg_shadow',
 'answer',
 'progress',
 'pg_roles',
 'incentive',
 'goal_progress',
 'goal',
 'pg_statistic_ext_data',
 'survey_availability',
 'calorie',
 'pg_hba_file_rules',
 'pg_settings',
 'pg_file_settings',
 'pg_backend_memory_contexts',
 'pg_ident_file_mappings',
 'speed',
 'pg_config',
 'pg_shmem_allocations',
 'pg_tables',
 'screentime',
 'screentime_app',
 'audio',
 'audio_response',
 'combined_data',
 'pg_user_mapping',
 'pg_statio_all_sequences',
 'pg_replication_origin_status',
 'pg_subscription',
 'pg_attribute',
 'pg_proc',
 'pg_class',
 'pg_attrdef',
 'pg_statio_sys_sequences',
 'pg_statio_user_sequences',
 'pg_constraint',
 'pg_inherits',
 'pg_index',
 'pg_operator',

Calorie table setup

In [9]:
calorie = db.extract_from_database("calorie")
calorie.head()
calorie.shape

/Users/pegah/Projects/lemurs-mqp/src/database_service.py:194: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.connection)
/Users/pegah/Projects/lemurs-mqp/src/database_service.py:218: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_users_df = pd.read_sql(query, self.connection)
2026-07-08 18:29:40,135 - INFO - Identified 5 test users to remove
2026-07-08 18:29:40,156 - INFO - Removed 2714 records associated with test users


(18554, 10)

Shape: (18554, 10)


In [10]:
calorie.columns

Index(['id', 'app_user_id', 'calories', 'start_timestamp', 'end_timestamp',
       'app_source', 'recorded_date', 'type', 'additional_data', 'status'],
      dtype='object')

In [11]:
calorie.head(10)

,id,app_user_id,calories,start_timestamp,end_timestamp,app_source,recorded_date,type,additional_data,status
0,1,6,14044,2025-11-18 18:04:25.157,2025-11-18 18:07:39.412,androidx.health.connect.client.records.metadat...,2025-11-18 18:09:34.050,calories,{},0
1,2,6,364283,2025-11-18 18:07:39.411,2025-11-19 00:00:00.000,androidx.health.connect.client.records.metadat...,2025-11-18 18:09:34.552,calories,{},0
2,3,9,1000,2025-11-19 14:15:00.000,2025-11-19 14:16:00.000,androidx.health.connect.client.records.metadat...,2025-11-19 14:27:36.364,calories,{},0
3,4,9,1000,2025-11-19 14:17:00.000,2025-11-19 14:18:00.000,androidx.health.connect.client.records.metadat...,2025-11-19 14:27:36.649,calories,{},0
4,5,9,1000,2025-11-19 14:19:00.000,2025-11-19 14:20:00.000,androidx.health.connect.client.records.metadat...,2025-11-19 14:27:36.860,calories,{},0
5,6,9,1000,2025-11-19 14:21:00.000,2025-11-19 14:22:00.000,androidx.health.connect.client.records.metadat...,2025-11-19 14:27:37.052,calories,{},0
6,7,9,1000,2025-11-19 14:23:00.000,2025-11-19 14:24:00.000,androidx.health.connect.client.records.metadat...,2025-11-19 14:27:37.250,calories,{},0
7,8,7,16014,2025-11-25 15:08:38.339,2025-11-25 15:12:53.662,androidx.health.connect.client.records.metadat...,2025-11-25 15:22:42.644,calories,{},0
8,9,7,23,2025-11-25 15:12:53.661,2025-11-25 15:12:55.234,androidx.health.connect.client.records.metadat...,2025-11-25 15:22:42.947,calories,{},0
9,10,7,4234,2025-11-25 15:12:55.233,2025-11-25 15:14:02.742,androidx.health.connect.client.records.metadat...,2025-11-25 15:22:43.258,calories,{},0


Calorie table data cleaning

1)

Data Cleaning to delete the rows with same start and end timestamps if any exists

In [12]:
same_time = calorie[
    calorie["start_timestamp"] == calorie["end_timestamp"]
]

print(f"Number of records with same start and end timestamps: {len(same_time)}")



Number of records with same start and end timestamps: 0


In [13]:
calorie = calorie[
    calorie["start_timestamp"] != calorie["end_timestamp"]
].copy()

In [14]:
calorie.shape

(18554, 10)

In [15]:
import pandas as pd
#converting start_timestamp and end_timestamp to datetime
calorie["start_timestamp"] = pd.to_datetime(calorie["start_timestamp"])
calorie["end_timestamp"] = pd.to_datetime(calorie["end_timestamp"])
calorie[["start_timestamp", "end_timestamp"]].head()
#calorie.dtypes

,start_timestamp,end_timestamp
0,2025-11-18 18:04:25.157,2025-11-18 18:07:39.412
1,2025-11-18 18:07:39.411,2025-11-19 00:00:00.000
2,2025-11-19 14:15:00.000,2025-11-19 14:16:00.000
3,2025-11-19 14:17:00.000,2025-11-19 14:18:00.000
4,2025-11-19 14:19:00.000,2025-11-19 14:20:00.000


In [16]:
calorie.shape

(18554, 10)

In [17]:
#calculating duration in minutes (start-end timestamps)

calorie['duration'] = (calorie['end_timestamp'] - calorie['start_timestamp']).dt.total_seconds() / 60
calorie[['start_timestamp', 'end_timestamp', 'duration']].head()


,start_timestamp,end_timestamp,duration
0,2025-11-18 18:04:25.157,2025-11-18 18:07:39.412,3.237583
1,2025-11-18 18:07:39.411,2025-11-19 00:00:00.000,352.343150
2,2025-11-19 14:15:00.000,2025-11-19 14:16:00.000,1.000000
3,2025-11-19 14:17:00.000,2025-11-19 14:18:00.000,1.000000
4,2025-11-19 14:19:00.000,2025-11-19 14:20:00.000,1.000000


---------------------------------------------------------

2.

Remove app_user_id = 44 (211 records) - Has start_timestamp ranging from 2022 - 2023 (calorie records only out of study window)


In [18]:
print((calorie["app_user_id"] == 44).sum())

211


In [19]:
calorie.shape

(18554, 11)

In [20]:
calorie = calorie[calorie["app_user_id"] != 44].copy()

In [21]:
calorie.shape

(18343, 11)

--------------------------------------------------------

3)

Identify cases with start_time stamp before september 2025 (study timeframe)


In [ ]:
pre2025=calorie[calorie["start_timestamp"] < "2025-09-01"][["start_timestamp", "end_timestamp", "duration", "calories", "app_user_id", "app_source"]]




In [23]:
calorie[calorie["start_timestamp"] < "2025-01-01"][["start_timestamp", "end_timestamp", "duration", "calories", "app_user_id", "app_source"]]

,start_timestamp,end_timestamp,duration,calories,app_user_id,app_source
1418,2024-11-13 18:45:15.190,2025-05-13 19:07:49.401,260662.570183,5027,22,Apple Watch
1420,2024-11-14 16:01:27.561,2024-11-15 14:42:17.137,1360.826267,2077,22,Apple Watch
1421,2024-11-13 19:00:15.882,2025-05-13 18:53:39.221,260633.388983,191,22,Apple Watch
1422,2024-11-15 14:42:17.137,2024-11-16 07:23:10.121,1000.883067,1517,22,Apple Watch
1423,2024-11-15 06:47:58.643,2024-11-15 17:08:15.427,620.279733,582,22,Apple Watch
...,...,...,...,...,...,...
17031,2024-12-29 09:06:27.002,2024-12-29 09:26:52.508,20.425100,36,35,Apple Watch
17035,2024-11-03 19:16:13.827,2024-11-04 12:34:33.716,1038.331483,250,35,Apple Watch
17036,2024-12-29 09:45:04.161,2024-12-29 10:15:39.698,30.592283,33,35,Apple Watch
17037,2024-11-03 18:52:01.627,2024-11-03 19:16:13.827,24.203333,26,35,Apple Watch


In [24]:
pre2025.groupby("app_user_id").size().sort_values(ascending=False)

app_user_id
22    1277
27    1216
35    1175
32       2
dtype: int64

In [25]:
#remove cases with start_time stamp before september 2025
calorie_cleaned = calorie[calorie["start_timestamp"] >= "2025-09-01"]
calorie_cleaned.shape

(14673, 11)

-------------------------------------------

4)

after the above data cleaning, app_user_id = 22 will have a total of 1 record --> remove  app_user_id = 22 


In [26]:

calorie_cleaned[calorie_cleaned["app_user_id"] == 22]

,id,app_user_id,calories,start_timestamp,end_timestamp,app_source,recorded_date,type,additional_data,status,duration
7108,7139,22,180,2026-03-14 18:08:20.677,2026-03-14 20:34:02.851,iPhone,2026-04-11 17:02:28.187,calories,{},0,145.7029


In [27]:
print((calorie_cleaned["app_user_id"] == 22).sum())

1


In [28]:
calorie_cleaned = calorie_cleaned[calorie_cleaned["app_user_id"] != 22].copy()

In [29]:

calorie_cleaned.shape


(14672, 11)

app_user_ids 22, 27, 35, 32 had records outside study duration.
After removing their out of study dates, only app_user_id 32 had other records in the study time stamp and the remaining (22, 27, 35) had 0 records.

*app_user id 27,35,44 can be removed when using both stap and calorie as they wont have calorie records inside the time frame of study (or their calorie can be marked as 0/missing.)

--------------------------------------------------------------

5)

Removing duplicate rows having the same [app_user_id, start_timestamp, end_timestamp, calories, app_source]


In [31]:
duplicate_count = calorie_cleaned.duplicated(
    subset=[
        "app_user_id",
        "start_timestamp",
        "end_timestamp",
        "calories",
        "app_source"
    ]
).sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 694


In [32]:
duplicates = calorie_cleaned[
    calorie_cleaned.duplicated(
        subset=[
            "app_user_id",
            "start_timestamp",
            "end_timestamp",
            "calories",
            "app_source"
        ],
        keep=False
    )
]

duplicates.sort_values(
    ["app_user_id", "start_timestamp"]
)

,id,app_user_id,calories,start_timestamp,end_timestamp,app_source,recorded_date,type,additional_data,status,duration
480,481,6,34491,2025-12-19 16:22:41.981,2025-12-19 16:30:39.062,androidx.health.connect.client.records.metadat...,2025-12-19 17:36:43.741,calories,{},0,7.951350
482,483,6,34491,2025-12-19 16:22:41.981,2025-12-19 16:30:39.062,androidx.health.connect.client.records.metadat...,2025-12-19 17:37:08.419,calories,{},0,7.951350
481,482,6,464576,2025-12-19 16:30:39.061,2025-12-20 00:00:00.000,androidx.health.connect.client.records.metadat...,2025-12-19 17:36:44.703,calories,{},0,449.348983
483,484,6,464576,2025-12-19 16:30:39.061,2025-12-20 00:00:00.000,androidx.health.connect.client.records.metadat...,2025-12-19 17:37:09.360,calories,{},0,449.348983
414,415,7,20,2025-12-15 13:28:02.879,2025-12-15 13:33:32.483,androidx.health.connect.client.records.metadat...,2025-12-15 14:48:23.632,calories,{},0,5.493400
...,...,...,...,...,...,...,...,...,...,...,...
20770,20801,49,0,2026-06-25 09:54:47.653,2026-06-25 09:55:08.026,iPhone,2026-06-25 10:12:41.249,calories,{},0,0.339550
21051,21082,49,51,2026-07-02 11:56:22.019,2026-07-02 12:44:17.018,iPhone,2026-07-02 12:51:19.471,calories,{},0,47.916650
21054,21085,49,51,2026-07-02 11:56:22.019,2026-07-02 12:44:17.018,iPhone,2026-07-02 12:51:19.689,calories,{},0,47.916650
21052,21083,49,7,2026-07-02 11:58:03.878,2026-07-02 12:37:06.703,iPhone,2026-07-02 12:51:19.613,calories,{},0,39.047083


In [33]:
duplicates.to_csv("cleaned_duplicate_rows_calorie.csv", index=False)

In [34]:
calorie_cleaned = calorie_cleaned.drop_duplicates(
    subset=[
        "app_user_id",
        "start_timestamp",
        "end_timestamp",
        "calories",
        "app_source"
    ],
    keep="first"
)

print("Shape after removing duplicates:", calorie_cleaned.shape)

Shape after removing duplicates: (13978, 11)


In [35]:
#checking if we have any duplicates now:
duplicate_rows = calorie_cleaned[
    calorie_cleaned.duplicated(
        subset=[
            "app_user_id",
            "start_timestamp",
            "end_timestamp",
            "calories",
            "app_source"
        ],
        keep=False
    )
]

print("Number of duplicate rows:", len(duplicate_rows))
duplicate_rows

Number of duplicate rows: 0


,id,app_user_id,calories,start_timestamp,end_timestamp,app_source,recorded_date,type,additional_data,status,duration


-----------------------------------------------------------------------------

6.

app_user_id = 21 has only 2 calorie records and 1 step record with app_source=HealthKit

Remove this user from study, was removed previously from step

Discontinued user

*This user is already in the globally drop indices*

In [36]:
calorie_cleaned[
    
    (calorie_cleaned["app_user_id"] == 21)
]

,id,app_user_id,calories,start_timestamp,end_timestamp,app_source,recorded_date,type,additional_data,status,duration
660,661,21,0,2026-03-03 18:13:36.873,2026-03-03 18:13:48.985,HealthKit,2026-03-03 18:15:36.422,calories,{},0,0.201867
661,662,21,16,2026-03-03 18:00:49.015,2026-03-03 18:13:48.985,HealthKit,2026-03-03 18:15:36.462,calories,{},0,12.999500


In [37]:
calorie_cleaned.shape

(13978, 11)

In [38]:
calorie_cleaned = calorie_cleaned[calorie_cleaned["app_user_id"] != 21].copy()

In [39]:
calorie_cleaned.shape

(13976, 11)

-----------------------------------------------------------------------------

6)

app_user_id = 6 has only 6 record in total for calorie.
and 18 records for step

-->remove user

In [40]:
calorie_cleaned = calorie_cleaned[calorie_cleaned["app_user_id"] != 6].copy()
calorie_cleaned.shape

(13972, 11)

In [41]:
calorie_cleaned[
    
    (calorie_cleaned["app_user_id"] == 6)
]

,id,app_user_id,calories,start_timestamp,end_timestamp,app_source,recorded_date,type,additional_data,status,duration


7.

For any android user with calorie recorded above or equal 3000, divide by 1000 (conversion) AND round to nearest integer

In [44]:
android_high_calorie = calorie_cleaned[
    calorie_cleaned["app_source"].str.contains("Android", case=False, na=False) &
    (calorie_cleaned["calories"] >= 3000)
]

print(android_high_calorie)

      id  app_user_id  calories         start_timestamp  \
7      8            7     16014 2025-11-25 15:08:38.339   
9     10            7      4234 2025-11-25 15:12:55.233   
10    11            7     12529 2025-11-25 15:14:02.741   
11    12            7    478602 2025-11-25 15:17:22.508   
13    14            9      3000 2025-11-26 07:32:00.000   
110  111            7     11935 2025-12-06 08:16:35.642   
111  112            7    861036 2025-12-06 08:19:45.928   
158  159            7    699879 2025-12-08 11:15:44.756   

              end_timestamp  \
7   2025-11-25 15:12:53.662   
9   2025-11-25 15:14:02.742   
10  2025-11-25 15:17:22.509   
11  2025-11-26 00:00:00.000   
13  2025-11-26 07:33:00.000   
110 2025-12-06 08:19:45.929   
111 2025-12-07 00:00:00.000   
158 2025-12-09 00:00:00.000   

                                            app_source  \
7    androidx.health.connect.client.records.metadat...   
9    androidx.health.connect.client.records.metadat...   
10   androidx.

In [45]:
android_high_calorie.shape

(8, 11)

In [46]:
calorie_cleaned.loc[
    calorie_cleaned["app_source"].str.contains("Android", case=False, na=False) &
    (calorie_cleaned["calories"] >= 3000),
    "calories"
] = (
    calorie_cleaned.loc[
        calorie_cleaned["app_source"].str.contains("Android", case=False, na=False) &
        (calorie_cleaned["calories"] >= 3000),
        "calories"
    ] / 1000
).round().astype(int)

In [ ]:
calorie_cleaned[
    calorie_cleaned["app_source"].str.contains("Android", case=False, na=False) &
    (calorie_cleaned["calories"] >= 3000)
]
#verify to have no calorie values greater than 3000 for Android users

,id,app_user_id,calories,start_timestamp,end_timestamp,app_source,recorded_date,type,additional_data,status,duration


In [49]:
calorie_cleaned.shape

(13972, 11)

--------------------------------------------------------------------

The final cleaned calorie data (including android, apple watch, iphone) has the shape of (13972, 11) up to the date July 8, 2026.

In [50]:
print("Records:", len(calorie_cleaned))
print("Users:", calorie_cleaned["app_user_id"].nunique())
calorie_cleaned["app_user_id"].unique()

Records: 13972
Users: 24


array([ 9,  7, 19, 20, 23, 24, 25, 26, 29, 28, 30, 31, 32, 33, 36, 37, 38,
       39, 40, 41, 46, 47, 48, 49])

---------------------------------------------------------------
-----------------------------------------------------------

**Creating different datasets for modeling:**

('calorie_cleaned' includes all the records after data cleaning)

Creating datasets including:

1. Only Android users
2. Only iPhone records (excluding Apple Watch records for users with both iPhone and Apple Watch)
3. Only IOS users (including Apple Watch records for users with both iPhone and Apple Watch, only exclude Android)

In [51]:
#1.only android users
android_calorie = calorie_cleaned[
    calorie_cleaned["app_source"].str.contains("android", case=False, na=False)
]

print("Records:", len(android_calorie))
print("Users:", android_calorie["app_user_id"].nunique())
android_calorie["app_user_id"].unique()

Records: 588
Users: 4


array([ 9,  7, 19, 20])

----------------------------------------------------

In [57]:
#2. Only iPhone users (excluding Apple Watch records for users with both iPhone and Apple Watch)


In [53]:
# Keep only iPhone records (for all users)
iphone_only_calorie = calorie_cleaned[
    calorie_cleaned["app_source"] == "iPhone"
]

print("Records:", len(iphone_only_calorie))
print("Users:", iphone_only_calorie["app_user_id"].nunique())
iphone_only_calorie["app_user_id"].unique()

Records: 9139
Users: 15


array([23, 25, 29, 28, 30, 31, 32, 33, 36, 38, 40, 46, 47, 48, 49])

------------------------------------------------------

In [54]:
#3. Only IOS users (including Apple Watch records for users with both iPhone and Apple Watch, only exclude Android)

ios_calorie = calorie_cleaned[
    calorie_cleaned["app_source"].isin(["iPhone", "Apple Watch"])
]

print("Records:", len(ios_calorie))
print("Users:", ios_calorie["app_user_id"].nunique())
ios_calorie["app_user_id"].unique()

Records: 13384
Users: 20


array([23, 24, 25, 26, 29, 28, 30, 31, 32, 33, 36, 37, 38, 39, 40, 41, 46,
       47, 48, 49])

--------------------------------------------------------

**All datasets created (4 total):**

1) 'calorie_clean': includes all the records after data cleaning (all Android and IOS (iPhone and Apple Watch records))
    - Records: 13972 - Users: 24
    - app_user_id: [ 9,  7, 19, 20, 23, 24, 25, 26, 29, 28, 30, 31, 32, 33, 36, 37, 38, 39, 40, 41, 46, 47, 48, 49]

2) 'android_calorie': includes only Android users 
    - Records: 588 - Users: 4
    - app_user_id: [7,  9, 19, 20]

3) 'iphone_only_calorie': includes only iPhone records (excluding Apple Watch records for users with both iPhone and Apple Watch)
    - Records: 9139 - Users: 15
    - app_user_id: [23, 25, 29, 28, 30, 31, 32, 33, 36, 38, 40, 46, 47, 48, 49]

4) 'ios_calorie': includes only IOS users (including Apple Watch records for users with both iPhone and Apple Watch, only exclude Android)
    - Records: 13384 - Users: 20
    - app_user_id: [23, 24, 25, 26, 29, 28, 30, 31, 32, 33, 36, 37, 38, 39, 40, 41, 46, 47, 48, 49]



--------------------------------------
-------------------------------------